# Bedtime Story AI Colab Workflow

Use this notebook for the repo at `https://github.com/aksakalai/bedtime_story_ai`.

Recommended update workflow after Codex pushes a change:
1. Run the `Update Repo` cell.
2. In Colab, click `Runtime -> Restart session`.
3. Run the `Relaunch After Restart` cell.

Important:
- Prefer `Restart session` over `Disconnect and delete runtime`.
- `Disconnect and delete runtime` destroys the VM and local model cache.
- `Restart session` clears Python and Gradio state without fully destroying the runtime.


In [ ]:
# OPTIONAL: CACHE HUGGING FACE DOWNLOADS TO GOOGLE DRIVE
# Run this if you want model files to survive full runtime deletion.
# If you do not need this, you can skip this cell.

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%env HF_HOME=/content/drive/MyDrive/bedtime_story_ai_cache/hf
%env HUGGINGFACE_HUB_CACHE=/content/drive/MyDrive/bedtime_story_ai_cache/hf/hub
%env TRANSFORMERS_CACHE=/content/drive/MyDrive/bedtime_story_ai_cache/hf/hub


In [ ]:
# FIRST-TIME SETUP FOR A FRESH COLAB RUNTIME
# Run this once after connecting to a new runtime.
# It clones the repo, installs the package, and prints the current commit.

%cd /content
!rm -rf bedtime_story_ai
!git clone https://github.com/aksakalai/bedtime_story_ai.git
%cd /content/bedtime_story_ai
!git rev-parse --short HEAD

!pip install -q --upgrade pip
!pip install -q -e .


In [ ]:
# LAUNCH THE APP AND PRELOAD MODELS
# Run this after the first-time setup cell.
# This loads the app, prints build/model info, preloads the models into memory,
# and launches the Gradio demo.

import gradio as gr
gr.close_all()

import story_app.app
import story_app.config

print('APP_BUILD:', story_app.config.APP_BUILD)
print('Description model:', story_app.config.DEFAULT_CONFIG.models.image_describer)
print('Story model:', story_app.config.DEFAULT_CONFIG.models.story_writer)

print(story_app.app.preload_models())

demo = story_app.app.build_demo()
demo.launch(debug=True, share=True, inline=False)


In [ ]:
# UPDATE THE LOCAL REPO AFTER CODEX PUSHES A CHANGE
# Run this when new code has been pushed to GitHub.
# This updates the repo on disk but does not relaunch the app yet.

%cd /content/bedtime_story_ai
!git checkout main
!git pull --ff-only origin main
!git rev-parse --short HEAD


## Restart Session Now

After running the update cell above:
- click `Runtime -> Restart session`
- do **not** click `Disconnect and delete runtime`

Then run the next cell.


In [ ]:
# RELAUNCH AFTER RESTARTING THE COLAB SESSION
# Run this after Runtime -> Restart session.
# This avoids the Gradio event-loop issue by starting a fresh Python process.

import gradio as gr
gr.close_all()

import story_app.app
import story_app.config

print('APP_BUILD:', story_app.config.APP_BUILD)
print('Description model:', story_app.config.DEFAULT_CONFIG.models.image_describer)
print('Story model:', story_app.config.DEFAULT_CONFIG.models.story_writer)

print(story_app.app.preload_models())

demo = story_app.app.build_demo()
demo.launch(debug=True, share=True, inline=False)


In [ ]:
# OPTIONAL: CLEAR LOADED MODELS FROM MEMORY
# Run this only if you explicitly want to free RAM or VRAM.

import story_app.app
print(story_app.app.clear_loaded_models())
